#cid and inichikey check.

In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

In [2]:
def canonicalize(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        return Chem.MolToSmiles(mol) if mol else None
    except Exception:
        return None

def smiles_to_inchikey(smi):
    csmi = canonicalize(smi)
    if csmi is None:
        return None
    mol = Chem.MolFromSmiles(csmi)
    try:
        key = inchi.MolToInchiKey(mol)
    except Exception:
        return None
    return key or None 




functions for both the summary assays and the NCATS assays:
- `load_and_tag_inchikey` reads a CSV and adds an `inchikey` column (duplicates are kept, nothing is dropped).
- `summarize_assay` reports row/column completeness for one assay.
- `find_cid_inchikey_conflicts` checks both directions: same CID -> different InChIKeys, and same InChIKey -> different CIDs.


In [3]:
def load_and_tag_inchikey(path, smiles_col='PUBCHEM_EXT_DATASOURCE_SMILES'):
    # Adds an 'inchikey' column computed from SMILES.
    # No duplicates are removed here 
    df = pd.read_csv(path, low_memory=False)
    df['inchikey'] = df[smiles_col].apply(smiles_to_inchikey)
    return df


def summarize_assay(df, label, cid_col='PUBCHEM_CID', smiles_col='PUBCHEM_EXT_DATASOURCE_SMILES'):
    return {
        'Assay': label,
        'total rows': len(df),
        'rows with CID': df[cid_col].notna().sum(),
        'rows with SMILES': df[smiles_col].notna().sum(),
        'rows with InChIKey': df['inchikey'].notna().sum(),
        'unique CIDs': df[cid_col].nunique(),
    }
#this function is used to find conflicts between CIDs and InChIKeys in a given dataframe. 
# It takes in a dataframe and the name of the column containing CIDs (default is 'PUBCHEM_CID').
#  It returns two dictionaries: one for CIDs that map to more than one InChIKey, and another for InChIKeys that map to more than one CID.
def find_cid_inchikey_conflicts(df, cid_col='PUBCHEM_CID'):
    # same CID -> more than one InChIKey, or same InChIKey -> more than one CID
    valid = df.dropna(subset=[cid_col, 'inchikey'])
    same_cid_diff_ikey = {cid: set(g) for cid, g in valid.groupby(cid_col)['inchikey'] if g.nunique() > 1}
    same_ikey_diff_cid = {ikey: set(g) for ikey, g in valid.groupby('inchikey')[cid_col] if g.nunique() > 1}
    return same_cid_diff_ikey, same_ikey_diff_cid

#this function is used to report the conflicts found in the data. 
# It takes in a label, two dictionaries containing the conflicts, and an optional id_label parameter. 
# It prints out a summary of the conflicts and returns a boolean indicating whether there are any conflicts or not.
def report_conflicts(label, same_cid_diff_ikey, same_ikey_diff_cid, id_label='CID'):
    ok = not same_cid_diff_ikey and not same_ikey_diff_cid
    print(f'{label}: {"IDENTICAL" if ok else "DIFFERS"}  '
          f'({id_label} -> >1 InChIKey: {len(same_cid_diff_ikey)}, '
          f'InChIKey -> >1 {id_label}: {len(same_ikey_diff_cid)})')
    for cid, ikeys in list(same_cid_diff_ikey.items())[:5]:
        print(f'    {id_label} {cid} -> InChIKeys {sorted(ikeys)}')
    for ikey, cids in list(same_ikey_diff_cid.items())[:5]:
        print(f'    InChIKey {ikey} -> {id_label}s {sorted(cids)}')
    return ok

print('Helpers ready.')


Helpers ready.


In [4]:
import numpy as np

def print_active_inactive(name, n_active, n_inactive):
    # one shared format for every split count printed in this notebook
    print(f'  {name:<12}: Active: {n_active:,}, Inactive: {n_inactive:,}')

In [5]:
BASE = '/Users/yamini/Desktop/projects/qsar casepase/qsar_prediction'
SUMMARY_DIR = f'{BASE}/Rebuttal/summary datasets'

three_summary_assays = {
    'AID_720637 (tox 21 mmp) ': f'{SUMMARY_DIR}/AID_720637_datatable (1).csv',
    'AID_1347034 (caspase hepg2)': f'{SUMMARY_DIR}/AID_1347034_datatable.csv',
    'AID_1347037 (caspase cho-k1) ': f'{SUMMARY_DIR}/AID_1347037_datatable.csv',
}
#note: merge ncats hepg2 and also tox 21 summary hepg2 

summary_dfs = {label: load_and_tag_inchikey(path) for label, path in three_summary_assays.items()}

summary_stats = pd.DataFrame([summarize_assay(df, label) for label, df in summary_dfs.items()]).set_index('Assay')
summary_stats


,total rows,rows with CID,rows with SMILES,rows with InChIKey,unique CIDs
Assay,,,,,
AID_720637 (tox 21 mmp),10490,10337,10337,10337,8099
AID_1347034 (caspase hepg2),9670,9524,9524,9524,7671
AID_1347037 (caspase cho-k1),9670,9524,9524,9524,7671


In [6]:
summary_results = {}
for label, df in summary_dfs.items():
    same_cid_diff_ikey, same_ikey_diff_cid = find_cid_inchikey_conflicts(df)
    summary_results[label] = report_conflicts(label, same_cid_diff_ikey, same_ikey_diff_cid)


AID_720637 (tox 21 mmp) : IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_1347034 (caspase hepg2): IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)


AID_1347037 (caspase cho-k1) : IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)




Rows where the same `PUBCHEM_CID` shows up more than once -- typically replicate
measurements of the same compound. All of them should share one `inchikey` per
CID (confirmed above: `CID -> >1 InChIKey` was 0).


In [7]:
def duplicate_cid_rows(df, cid_col='PUBCHEM_CID'):
    # rows for CIDs that appear more than once, so you can see whether the
    # repeats agree on InChIKey (same compound, replicate measurement) or not
    sizes = df.groupby(cid_col).size()
    dup_cids = sizes[sizes > 1].index
    cols = [cid_col, 'PUBCHEM_EXT_DATASOURCE_SMILES', 'inchikey',
            'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']
    cols = [c for c in cols if c in df.columns]
    return df[df[cid_col].isin(dup_cids)].sort_values(cid_col)[cols]


In [8]:
aid_key = next(k for k in summary_dfs if k.strip().startswith('AID_1347034'))
duplicate_cid_rows(summary_dfs[aid_key])


,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,inchikey,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
853,6.0,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,VYZAHLCBVHPDDF-UHFFFAOYSA-N,Inconclusive,5.0
681,6.0,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,VYZAHLCBVHPDDF-UHFFFAOYSA-N,Inconclusive,20.0
3346,13.0,C1=CC(=C(C=C1Cl)Cl)Cl,PBKONEOXTCPAFI-UHFFFAOYSA-N,Inactive,0.0
7500,13.0,C1=CC(=C(C=C1Cl)Cl)Cl,PBKONEOXTCPAFI-UHFFFAOYSA-N,Inactive,0.0
8020,66.0,C1=CC(=C(C=C1Cl)O)Cl,RANCECPPZPIPNO-UHFFFAOYSA-N,Inactive,0.0
...,...,...,...,...,...
5197,135465076.0,C1=CC=C(C=C1)/[N+](=N/[O-])/[O-].[NH4+],GXCSNALCLRPEAS-CFYXSCKTSA-N,Inactive,0.0
7668,135465076.0,C1=CC=C(C=C1)/[N+](=N/[O-])/[O-].[NH4+],GXCSNALCLRPEAS-CFYXSCKTSA-N,Inactive,0.0
7569,136122621.0,C[C@H]1/C=C\C=C(/C(=O)NC2=C(C(=C3C(=C2O)C(=C(C...,JQXXHWHPUNPDRT-CHTVYFSYSA-N,Inactive,0.0
2476,136122621.0,C[C@H]1/C=C\C=C(/C(=O)NC2=C(C(=C3C(=C2O)C(=C(C...,JQXXHWHPUNPDRT-CHTVYFSYSA-N,Inactive,0.0


In [9]:
def summarize_balance(df, label, cid_col='PUBCHEM_CID', outcome_col='PUBCHEM_ACTIVITY_OUTCOME'):
    # one row per compound -- CID <-> InChIKey is already confirmed 1:1 for these
    # files, so deduping by CID also gives one row per unique structure
    per_compound = df.dropna(subset=[cid_col]).drop_duplicates(subset=[cid_col])
    counts = per_compound[outcome_col].value_counts()
    n_active       = int(counts.get('Active', 0))
    n_inactive     = int(counts.get('Inactive', 0))
    n_inconclusive = int(counts.get('Inconclusive', 0))
    n_tested = n_active + n_inactive  # Inconclusive excluded from the ratio
    ratio = f'1:{n_inactive / n_active:.1f}' if n_active else 'n/a'
    return {
        'Assay': label,
        'compounds': len(per_compound),
        'Active': n_active,
        'Inactive': n_inactive,
        'Inconclusive': n_inconclusive,
        'Imbalance (Inactive:Active)': ratio,
        '% Active (of Active+Inactive)': round(100 * n_active / n_tested, 2) if n_tested else float('nan'),
    }

In [10]:
balance_stats = pd.DataFrame([summarize_balance(df, label) for label, df in summary_dfs.items()]).set_index('Assay')
balance_stats

,compounds,Active,Inactive,Inconclusive,Imbalance (Inactive:Active),% Active (of Active+Inactive)
Assay,,,,,,
AID_720637 (tox 21 mmp),8099,901,5315,1883,1:5.9,14.49
AID_1347034 (caspase hepg2),7671,353,6968,350,1:19.7,4.82
AID_1347037 (caspase cho-k1),7671,193,7325,153,1:38.0,2.57


## 2. The 13 NCATS caspase assays

In [11]:
ncats_aids = [654, 655, 656, 657, 658, 659, 660, 661, 663, 664, 665, 666, 667]
ncats_assays = {f'AID_{a}': f'{BASE}/cleaned_datasets/AID_{a}_cleaned.csv' for a in ncats_aids}

ncats_dfs = {label: load_and_tag_inchikey(path) for label, path in ncats_assays.items()}

ncats_stats = pd.DataFrame([summarize_assay(df, label) for label, df in ncats_dfs.items()]).set_index('Assay')
ncats_stats


,total rows,rows with CID,rows with SMILES,rows with InChIKey,unique CIDs
Assay,,,,,
AID_654,1333,1333,1333,1333,1333
AID_655,1333,1333,1333,1333,1333
AID_656,1333,1333,1333,1333,1333
AID_657,1333,1333,1333,1333,1333
AID_658,1333,1333,1333,1333,1333
AID_659,1334,1334,1334,1334,1334
AID_660,1334,1334,1334,1334,1334
AID_661,1334,1334,1334,1334,1334
AID_663,1333,1333,1333,1333,1333


In [12]:
duplicate_cid_rows(ncats_dfs['AID_665'])

,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,inchikey,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE


In [13]:
ncats_results = {}
for label, df in ncats_dfs.items():
    same_cid_diff_ikey, same_ikey_diff_cid = find_cid_inchikey_conflicts(df)
    ncats_results[label] = report_conflicts(label, same_cid_diff_ikey, same_ikey_diff_cid)


AID_654: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_655: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_656: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_657: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_658: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)


AID_659: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_660: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_661: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_663: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_664: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)


AID_665: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_666: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)
AID_667: IDENTICAL  (CID -> >1 InChIKey: 0, InChIKey -> >1 CID: 0)


## 3. DILI liver datasets

These files have no PubChem CID -- just `smiles_r` (+ `TOXICITY` for most). Canonical
SMILES (`csmi`) plays the role CID played for the assay files: check whether grouping
by `csmi` agrees with grouping by `inchikey`, and whether each file already has
duplicate compounds within it.


In [14]:
dili_datasets = {
    'Gold-1111':   f'{BASE}/liver/DILI_Goldstandard_1111.csv',
    'Held-223':    f'{BASE}/liver/test_data_heldouttest_DILIst_223.csv',
    'Alloriginal': f'{BASE}/liver/Alloriginaldata_19911.csv',
    'NCV-888':     f'{BASE}/liver/test_data_ncv_DILIst_888.csv',
}

dili_dfs = {}
for label, path in dili_datasets.items():
    df = load_and_tag_inchikey(path, smiles_col='smiles_r')
    df['csmi'] = df['smiles_r'].apply(canonicalize)  # stands in for CID here
    dili_dfs[label] = df


/var/folders/yp/d7h16rp15994f_chv77wlj0w0000gn/T/ipykernel_1695/3715028867.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['inchikey'] = df[smiles_col].apply(smiles_to_inchikey)
/var/folders/yp/d7h16rp15994f_chv77wlj0w0000gn/T/ipykernel_1695/3282693694.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['csmi'] = df['smiles_r'].apply(canonicalize)  # stands in for CID here


/var/folders/yp/d7h16rp15994f_chv77wlj0w0000gn/T/ipykernel_1695/3715028867.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['inchikey'] = df[smiles_col].apply(smiles_to_inchikey)
/var/folders/yp/d7h16rp15994f_chv77wlj0w0000gn/T/ipykernel_1695/3282693694.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['csmi'] = df['smiles_r'].apply(canonicalize)  # stands in for CID here


In [15]:
dili_stats = pd.DataFrame(
    [summarize_assay(df, label, cid_col='csmi', smiles_col='smiles_r') for label, df in dili_dfs.items()]
).set_index('Assay').rename(columns={'rows with CID': 'rows with csmi', 'unique CIDs': 'unique csmi'})
dili_stats


,total rows,rows with csmi,rows with SMILES,rows with InChIKey,unique csmi
Assay,,,,,
Gold-1111,1111,1096,1111,1096,1096
Held-223,223,222,223,222,222
Alloriginal,19911,19911,19911,19908,13788
NCV-888,888,874,888,874,874


In [156]:
dili_results = {}
for label, df in dili_dfs.items():
    same_csmi_diff_ikey, same_ikey_diff_csmi = find_cid_inchikey_conflicts(df, cid_col='csmi')
    dili_results[label] = report_conflicts(label, same_csmi_diff_ikey, same_ikey_diff_csmi, id_label='csmi')


Gold-1111: IDENTICAL  (csmi -> >1 InChIKey: 0, InChIKey -> >1 csmi: 0)
Held-223: IDENTICAL  (csmi -> >1 InChIKey: 0, InChIKey -> >1 csmi: 0)
Alloriginal: DIFFERS  (csmi -> >1 InChIKey: 0, InChIKey -> >1 csmi: 625)
    InChIKey AAEVYOVXGOFMJO-UHFFFAOYSA-N -> csmis ['CSc1nc(=NC(C)C)[nH]c(=NC(C)C)[nH]1', 'CSc1nc(NC(C)C)nc(NC(C)C)n1']
    InChIKey ACGDKVXYNVEAGU-UHFFFAOYSA-N -> csmis ['N=C(N)NCCN1CCCCCCC1', 'NC(N)=NCCN1CCCCCCC1']
    InChIKey ACWBQPMHZXGDFX-QFIPXVFZSA-N -> csmis ['CCCCC(=O)N(Cc1ccc(-c2ccccc2-c2nn[nH]n2)cc1)[C@H](C(=O)O)C(C)C', 'CCCCC(=O)N(Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1)[C@H](C(=O)O)C(C)C']
    InChIKey ACWBQPMHZXGDFX-UHFFFAOYSA-N -> csmis ['CCCCC(=O)N(Cc1ccc(-c2ccccc2-c2nn[nH]n2)cc1)C(C(=O)O)C(C)C', 'CCCCC(=O)N(Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1)C(C(=O)O)C(C)C']
    InChIKey AEOCXXJPGCBFJA-UHFFFAOYSA-N -> csmis ['CCc1cc(C(=N)S)ccn1', 'CCc1cc(C(N)=S)ccn1']
NCV-888: IDENTICAL  (csmi -> >1 InChIKey: 0, InChIKey -> >1 csmi: 0)


### InChIKey conflict rows (Alloriginal example)

Rows whose `inchikey` is shared by more than one distinct `csmi` -- sorted by
`inchikey` so the different SMILES spellings of the same compound sit next to
each other.


In [157]:
def inchikey_conflict_rows(df, id_col='csmi'):
    # rows whose InChIKey is shared by more than one distinct id_col value
    _, conflicts = find_cid_inchikey_conflicts(df, cid_col=id_col)
    cols = [c for c in [id_col, 'inchikey', 'smiles_r', 'TOXICITY'] if c in df.columns]
    return df[df['inchikey'].isin(conflicts)].sort_values('inchikey')[cols]


In [158]:
inchikey_conflict_rows(dili_dfs['Alloriginal'])


,csmi,inchikey,smiles_r,TOXICITY
2976,CSc1nc(NC(C)C)nc(NC(C)C)n1,AAEVYOVXGOFMJO-UHFFFAOYSA-N,CSc1nc(NC(C)C)nc(NC(C)C)n1,0
5674,CSc1nc(NC(C)C)nc(NC(C)C)n1,AAEVYOVXGOFMJO-UHFFFAOYSA-N,CSc1nc(NC(C)C)nc(NC(C)C)n1,1
3501,CSc1nc(NC(C)C)nc(NC(C)C)n1,AAEVYOVXGOFMJO-UHFFFAOYSA-N,CSc1nc(NC(C)C)nc(NC(C)C)n1,0
16611,CSc1nc(=NC(C)C)[nH]c(=NC(C)C)[nH]1,AAEVYOVXGOFMJO-UHFFFAOYSA-N,CSc1nc(=NC(C)C)[nH]c(=NC(C)C)[nH]1,0
7993,NC(N)=NCCN1CCCCCCC1,ACGDKVXYNVEAGU-UHFFFAOYSA-N,NC(N)=NCCN1CCCCCCC1,0
...,...,...,...,...
19908,CCOC(=O)CN[C@@H](C(=O)N1CC[C@H]1C(=O)NCc1ccc(/...,ZXIBCJHYVWYIKI-PZJWPPBQSA-N,CCOC(=O)CN[C@@H](C(=O)N1CC[C@H]1C(=O)NCc1ccc(/...,0
118,CCOC(=O)CNC(C(=O)N1CCC1C(=O)NCc1ccc(/C(N)=N/O)...,ZXIBCJHYVWYIKI-UHFFFAOYSA-N,CCOC(=O)CNC(C(=O)N1CCC1C(=O)NCc1ccc(/C(N)=N/O)...,1
7534,CCOC(=O)CNC(C(=O)N1CCC1C(=O)NCc1ccc(C(N)=NO)cc...,ZXIBCJHYVWYIKI-UHFFFAOYSA-N,CCOC(=O)CNC(C(=O)N1CCC1C(=O)NCc1ccc(C(N)=NO)cc...,1
15686,CN(C)c1ccc(O)c2c1C[C@H]1C[C@H]3[C@H](N(C)C)C(=...,ZZZRUAITSXLWBH-CSYJEMQYSA-N,CN(C)c1ccc(O)c2c1C[C@H]1C[C@H]3[C@H](N(C)C)C(=...,0


In [154]:
# Gold-1111 = NCV-888 + Held-223 combined, so removing Gold-1111 from Alloriginal
# should remove both. Remove by csmi (what the pipeline actually does), then check by
# InChIKey whether any of 888 or 223 still intersect what's left -- i.e. still leaking.
gold_csmi = set(dili_dfs['Gold-1111']['csmi'].dropna())
clin_pool = dili_dfs['Alloriginal'][~dili_dfs['Alloriginal']['csmi'].isin(gold_csmi)]

clin_ikeys = set(clin_pool['inchikey'].dropna())
ncv_ikeys  = set(dili_dfs['NCV-888']['inchikey'].dropna())
held_ikeys = set(dili_dfs['Held-223']['inchikey'].dropna())

print(f'Clinical pool ∩ NCV-888  (InChIKey): {len(clin_ikeys & ncv_ikeys)}')
print(f'Clinical pool ∩ Held-223 (InChIKey): {len(clin_ikeys & held_ikeys)}')


Clinical pool ∩ NCV-888  (InChIKey): 174
Clinical pool ∩ Held-223 (InChIKey): 55


In [160]:
# Dedup = keep only ONE row per real molecule (same InChIKey = same compound, different csmi spelling merged).
from IPython.display import display

dili_dfs_dedup = {}
for label, df in dili_dfs.items():
    before = len(df)
    deduped = df.drop_duplicates('inchikey').reset_index(drop=True)
    after = len(deduped)
    dili_dfs_dedup[label] = deduped
    print(f'{label}: {before} rows -> {after} unique compounds (InChIKey dedup, {before - after} rows removed)')

# Pick a few compounds that had duplicate rows in Alloriginal, and show
# what they looked like BEFORE dedup vs the single row kept AFTER dedup.
df = dili_dfs['Alloriginal']
ikey_counts = df['inchikey'].value_counts()
example_ikeys = ikey_counts[ikey_counts > 1].head(3).index
cols = [c for c in ['csmi', 'inchikey', 'smiles_r', 'TOXICITY'] if c in df.columns]

print('\nBEFORE dedup -- same InChIKey, multiple rows:')
display(df[df['inchikey'].isin(example_ikeys)].sort_values('inchikey')[cols])

print('AFTER dedup -- one row kept per InChIKey:')
after_df = dili_dfs_dedup['Alloriginal']
display(after_df[after_df['inchikey'].isin(example_ikeys)].sort_values('inchikey')[cols])


Gold-1111: 1111 rows -> 1097 unique compounds (InChIKey dedup, 14 rows removed)
Held-223: 223 rows -> 223 unique compounds (InChIKey dedup, 0 rows removed)
Alloriginal: 19911 rows -> 13123 unique compounds (InChIKey dedup, 6788 rows removed)
NCV-888: 888 rows -> 875 unique compounds (InChIKey dedup, 13 rows removed)

BEFORE dedup -- same InChIKey, multiple rows:


,csmi,inchikey,smiles_r,TOXICITY
4739,Sc1[nH]cnc2ncnc1-2,GLVAUDGFNGKCSF-UHFFFAOYSA-N,Sc1[nH]cnc2ncnc1-2,1
9264,S=c1nc[nH]c2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1nc[nH]c2nc[nH]c12,1
9977,S=c1nc[nH]c2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1nc[nH]c2nc[nH]c12,0
7519,S=c1nc[nH]c2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1nc[nH]c2nc[nH]c12,1
7403,Sc1ncnc2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,Sc1ncnc2nc[nH]c12,1
9519,S=c1nc[nH]c2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1nc[nH]c2nc[nH]c12,1
12963,Sc1ncnc2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,Sc1ncnc2nc[nH]c12,0
9646,Sc1ncnc2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,Sc1ncnc2nc[nH]c12,1
1311,S=c1nc[nH]c2nc[nH]c12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1nc[nH]c2nc[nH]c12,1
370,S=c1[nH]cnc2[nH]cnc12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1[nH]cnc2[nH]cnc12,1


AFTER dedup -- one row kept per InChIKey:


,csmi,inchikey,smiles_r,TOXICITY
370,S=c1[nH]cnc2[nH]cnc12,GLVAUDGFNGKCSF-UHFFFAOYSA-N,S=c1[nH]cnc2[nH]cnc12,1
1329,Nc1nc(=O)c2ncn(COC(CO)CO)c2[nH]1,IRSCQMHQWWYFCW-UHFFFAOYSA-N,Nc1nc(=O)c2ncn(COC(CO)CO)c2[nH]1,1
360,Oc1ncnc2[nH]ncc12,OFCNXPDARWKPPY-UHFFFAOYSA-N,Oc1ncnc2[nH]ncc12,1


In [ ]:
import os

OUT_DIR = f'{BASE}/Rebuttal/results'
os.makedirs(OUT_DIR, exist_ok=True)

# Summary tables from each section
summary_stats.to_csv(f'{OUT_DIR}/summary_assays_stats.csv')
ncats_stats.to_csv(f'{OUT_DIR}/ncats_assays_stats.csv')
balance_stats.to_csv(f'{OUT_DIR}/summary_assays_balance.csv')
dili_stats.to_csv(f'{OUT_DIR}/dili_stats.csv')

# Dedup before/after counts, one row per DILI dataset
dedup_summary = pd.DataFrame([
    {'Assay': label, 'rows before': len(dili_dfs[label]), 'rows after': len(dili_dfs_dedup[label]),
     'rows removed': len(dili_dfs[label]) - len(dili_dfs_dedup[label])}
    for label in dili_dfs
]).set_index('Assay')
dedup_summary.to_csv(f'{OUT_DIR}/dili_dedup_summary.csv')

# The InChIKey conflict rows found in Alloriginal
inchikey_conflict_rows(dili_dfs['Alloriginal']).to_csv(f'{OUT_DIR}/alloriginal_inchikey_conflicts.csv', index=False)

# Deduplicated DILI datasets themselves
for label, df in dili_dfs_dedup.items():
    safe_name = label.replace(' ', '_')
    df.to_csv(f'{OUT_DIR}/{safe_name}_deduped.csv', index=False)

print(f'Saved results to: {OUT_DIR}')
for f in sorted(os.listdir(OUT_DIR)):
    print(' -', f)


## 4. Salt-stripped InChIKey check -- MMP summary assay (AID 720637)

Section 1 confirmed CID <-> InChIKey is 1:1 for AID_720637, but that InChIKey was
computed on the SMILES as-is, salt/counterion still attached (`smiles_to_inchikey`
in cell 2 never strips anything). Two different CIDs -- one a free base, one a salt
of the exact same active molecule -- get two different InChIKeys under that scheme,
so this specific check cannot see them as duplicates.

Here we add a second InChIKey computed on the LARGEST FRAGMENT only (salts/counterions
removed first), and check whether the CID count changes once compounds are deduped on
that instead.

In [28]:
def smiles_to_inchikey_stripped(smi):
    # same as smiles_to_inchikey, but keeps only the largest fragment
    # (by heavy-atom count) before computing the key -- standard salt-strip heuristic
    csmi = canonicalize(smi)
    if csmi is None:
        return None
    mol = Chem.MolFromSmiles(csmi)
    if mol is None:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    if len(frags) > 1:
        mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())
    try:
        key = inchi.MolToInchiKey(mol)
    except Exception:
        return None
    return key or None


mmp_key = next(k for k in summary_dfs if k.strip().startswith('AID_720637'))
mmp_df = summary_dfs[mmp_key].copy()
mmp_df['inchikey_stripped'] = mmp_df['PUBCHEM_EXT_DATASOURCE_SMILES'].apply(smiles_to_inchikey_stripped)

n_cids_before = mmp_df['PUBCHEM_CID'].nunique()
n_ikey_before = mmp_df['inchikey'].nunique()               # as in section 1, salt attached
n_ikey_stripped = mmp_df['inchikey_stripped'].nunique()    # salt removed first

print(f'{mmp_key}')
print(f'  unique CIDs                          : {n_cids_before:,}')
print(f'  unique InChIKeys (no strip, section 1): {n_ikey_before:,}')
print(f'  unique InChIKeys (salt-stripped)      : {n_ikey_stripped:,}')
print(f'  compounds that collapse once stripped : {n_ikey_before - n_ikey_stripped:,}')

AID_720637 (tox 21 mmp) 
  unique CIDs                          : 8,099
  unique InChIKeys (no strip, section 1): 8,099
  unique InChIKeys (salt-stripped)      : 7,691
  compounds that collapse once stripped : 408


In [28]:
# CIDs that collapse onto the same stripped InChIKey -- i.e. same real molecule,
# different CID because a counterion is present in one and not the other
valid = mmp_df.dropna(subset=['PUBCHEM_CID', 'inchikey_stripped'])
same_stripped_ikey_diff_cid = {
    ikey: sorted(set(g)) for ikey, g in valid.groupby('inchikey_stripped')['PUBCHEM_CID'] if g.nunique() > 1
}
print(f'Distinct real molecules represented by >1 CID once salts are stripped: {len(same_stripped_ikey_diff_cid)}')

cols = ['PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'inchikey', 'inchikey_stripped', 'PUBCHEM_ACTIVITY_OUTCOME']
example_ikeys = list(same_stripped_ikey_diff_cid.keys())[:5]
mmp_df[mmp_df['inchikey_stripped'].isin(example_ikeys)].sort_values('inchikey_stripped')[cols]

Distinct real molecules represented by >1 CID once salts are stripped: 242
